# 8-Agent Enterprise RAG Architecture with Human-in-the-Loop

This notebook runs the complete 8-agent proposal generation workflow.

## Features
- ✅ Automated RFP matching via semantic search
- ✅ Template selection (automatic or manual)
- ✅ Content generation from ALL proposals (RAG)
- ✅ Format extraction from ONE matched proposal
- ✅ Governance calculator (SteerCo and weekly meetings)
- ✅ Human-in-the-loop at 4 decision points

## Architecture
- **Track 1 (Content)**: Agents 3-6 learn from ALL proposals
- **Track 2 (Format)**: Agents 1-2, 7-8 use ONE matched proposal
- **HITL**: Human can review and override at each stage

## Setup

In [ ]:
# Import libraries
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Load environment
from dotenv import load_dotenv
load_dotenv()

# Check API key
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("❌ ERROR: OPENAI_API_KEY not found in .env file")
else:
    print("✅ OpenAI API key loaded")
    print(f"   Key: {api_key[:15]}...")

## Check Data Files

In [ ]:
# Check for RFP and Proposal files
data_dir = Path.cwd().parent / 'data'
rfp_dir = data_dir / 'rfps'
proposal_dir = data_dir / 'proposals'

rfp_files = list(rfp_dir.glob('*.pdf')) if rfp_dir.exists() else []
proposal_files = list(proposal_dir.glob('*.pptx')) if proposal_dir.exists() else []

print(f"📁 Data Files:")
print(f"   RFP PDFs: {len(rfp_files)}")
for f in rfp_files[:5]:
    print(f"      - {f.name}")
if len(rfp_files) > 5:
    print(f"      ... and {len(rfp_files) - 5} more")

print(f"\n   Proposal PPTXs: {len(proposal_files)}")
for f in proposal_files[:5]:
    print(f"      - {f.name}")
if len(proposal_files) > 5:
    print(f"      ... and {len(proposal_files) - 5} more")

if not rfp_files or not proposal_files:
    print("\n⚠️  Warning: Add more files for better results")

## Initialize RAG Engine

In [ ]:
from rag_engine import RAGEngine

print("🔄 Initializing RAG Engine...")
rag = RAGEngine(
    api_key=api_key,
    db_path="../data/chroma_db",
    verbose=False
)

# Check if database exists
db_path = Path.cwd().parent / 'data' / 'chroma_db'
if not db_path.exists():
    print("\n⚠️  Database not indexed yet!")
    print("   Run: python index_database.py")
else:
    stats = rag.get_collection_stats()
    print(f"\n✅ RAG Engine initialized")
    print(f"   Total chunks: {stats.get('total_chunks', 0)}")
    print(f"   RFP chunks: {stats.get('doc_types', {}).get('rfp', 0)}")
    print(f"   Proposal chunks: {stats.get('doc_types', {}).get('proposal', 0)}")

## Option 1: Run Complete Workflow (Programmatic)

In [ ]:
# Import workflow function
sys.path.insert(0, str(Path.cwd().parent))
from main_workflow import generate_proposal

# Choose RFP
rfp_path = str(rfp_files[0]) if rfp_files else None

if not rfp_path:
    print("❌ No RFP files found. Add PDFs to data/rfps/")
else:
    print(f"📄 Using RFP: {Path(rfp_path).name}")
    
    # Run workflow (automated mode - AI makes all decisions)
    output_path = generate_proposal(
        rfp_path=rfp_path,
        output_path=f"../output/{Path(rfp_path).stem}_proposal.pptx",
        interactive=False,  # Set to True for HITL mode
        verbose=True
    )
    
    if output_path:
        print(f"\n✅ Proposal generated: {output_path}")

## Option 2: Step-by-Step with Manual Control

Run each agent individually for maximum control.

### Step 1: Parse RFP

In [ ]:
from rfp_parser import parse_rfp

# Choose RFP file
rfp_path = str(rfp_files[0]) if rfp_files else None

if not rfp_path:
    print("❌ No RFP files found")
else:
    rfp_data = parse_rfp(rfp_path, verbose=True)
    rfp_text = rfp_data['full_text']
    project_name = rfp_data['project_name']
    
    print(f"\n📊 RFP Summary:")
    print(f"   Project: {project_name}")
    print(f"   Pages: {rfp_data['page_count']}")
    print(f"   Text length: {len(rfp_text):,} characters")
    print(f"\n   First 500 characters:")
    print(f"   {rfp_text[:500]}...")

### Step 2: Agent 1 - RFP Matcher

In [ ]:
from rfp_matcher import RFPMatcher

rfp_matcher = RFPMatcher(rag_engine=rag, verbose=True)
matched_rfp = rfp_matcher.find_similar_rfp(rfp_text)

if matched_rfp:
    print(f"\n✅ Best Match:")
    print(f"   File: {matched_rfp.rfp_file}")
    print(f"   Similarity Score: {matched_rfp.similarity_score:.3f}")
    print(f"   Evidence: {matched_rfp.evidence_count} matching chunks")
else:
    print("\n⚠️  No similar RFP found")

### Step 3: Agent 2 - Template Selector

In [ ]:
from template_selector import TemplateSelector

template_selector = TemplateSelector(
    proposals_dir="../data/proposals",
    verbose=True
)

if matched_rfp:
    template_file = template_selector.map_rfp_to_proposal(matched_rfp.rfp_file)
    
    if template_file:
        print(f"\n✅ Template Selected: {Path(template_file).name}")
    else:
        print("\n⚠️  No matching template found")
        print("\n   Available templates:")
        for i, p in enumerate(proposal_files, 1):
            print(f"   [{i}] {p.name}")
else:
    template_file = None

### Step 4: Agents 3-6 - Content Generation

In [ ]:
from content_generator import ContentGenerator
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=api_key,
    temperature=0.3
)

content_gen = ContentGenerator(
    rag_engine=rag,
    llm=llm,
    verbose=True
)

project_structure = content_gen.generate_content(
    rfp_text=rfp_text,
    project_name=project_name,
    duration_weeks=None
)

print(f"\n✅ Content Generated:")
print(f"   Workstreams: {len(project_structure.workstreams)}")
print(f"   Total Modules: {sum(len(ws.modules) for ws in project_structure.workstreams)}")
print(f"   Dependencies: {len(project_structure.dependencies.workstream_dependencies)}")

### Step 5: Governance Calculator (NEW!)

In [ ]:
from governance_calculator import GovernanceCalculator

gov_calc = GovernanceCalculator(rag_engine=rag, verbose=True)

total_duration = sum(ws.duration_weeks for ws in project_structure.workstreams)

governance = gov_calc.calculate_governance(
    workstreams=project_structure.workstreams,
    project_duration_weeks=total_duration
)

project_structure.governance = governance

### Step 6: Agent 7 - Template Extractor

In [ ]:
from template_extractor import TemplateExtractor

if template_file:
    extractor = TemplateExtractor(verbose=True)
    template_blueprint = extractor.extract_template(template_file)
    
    print(f"\n✅ Template extracted from: {template_blueprint['source_file']}")
else:
    print("\n⚠️  Using default template")
    from pptx.util import Inches
    template_blueprint = {
        'source_file': 'default',
        'dimensions': {
            'width': Inches(10),
            'height': Inches(5.625),
            'ratio': 16/9,
            'is_16_9': True
        },
        'timeline_template': None,
        'governance_template': None,
        'module_template': None,
        'workstream_template': None
    }

### Step 7: Agent 8 - Template Replicator

In [ ]:
from approach_replicator import ApproachSlideReplicator

replicator = ApproachSlideReplicator(
    templates=template_blueprint,
    verbose=True
)

presentation = replicator.create_approach_presentation(project_structure)

print(f"\n✅ Presentation created: {len(presentation.slides)} slides")

### Step 8: Save Output

In [ ]:
# Save presentation
output_dir = Path.cwd().parent / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{project_name}_proposal.pptx"
presentation.save(str(output_path))

print(f"\n✅ COMPLETE!")
print(f"\n📊 Output saved to: {output_path}")
print(f"\n   Slides: {len(presentation.slides)}")
print(f"   Workstreams: {len(project_structure.workstreams)}")
print(f"   SteerCo meetings: {governance['steerco']['total_meetings']}")
print(f"   Weekly meetings: {governance['weekly']['total_meetings']}")

## Review Generated Structure

In [ ]:
# Display project structure
print("📊 PROJECT STRUCTURE")
print("="*80)

for ws in project_structure.workstreams:
    print(f"\n🎯 {ws.name}")
    print(f"   Duration: {ws.duration_weeks} weeks")
    print(f"   Modules: {len(ws.modules)}")
    
    for i, mod in enumerate(ws.modules, 1):
        print(f"\n   {i}. {mod.name}")
        print(f"      Duration: {mod.duration_weeks} weeks")
        print(f"      Activities: {len(mod.activities)}")
        print(f"      Deliverables: {len(mod.deliverables)}")

print(f"\n{'='*80}")
print(f"\n🎯 GOVERNANCE")
print(f"   SteerCo: {governance['steerco']['total_meetings']} meetings ({governance['steerco']['cadence_description']})")
print(f"   Weekly: {governance['weekly']['total_meetings']} meetings ({governance['weekly']['cadence_description']})")